## Prediction dashboard

The newest prediction maps, and how well the predictions of the last `lookback_hours` matched what happened.

A prediction's outcome is known about 40 minutes after its poll (the 30-minute horizon plus the check poll),
so the newest predictions have no outcome yet and are not in the performance figures.

This notebook only reads tables; it writes nothing.

In [0]:
import os
import sys
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from sklearn.metrics import roc_auc_score

sys.path.append(os.path.abspath("../free_edition_models"))   # station_features.py is in free_edition_models
from station_features import build_predictors, build_targets
from station_plots import plot_prediction_maps, add_figure_heading

import io
import boto3

# Scheduled runs pass run_mode = "production" and publish the chart. Interactive runs only show it.
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
IS_PRODUCTION = dbutils.widgets.get("run_mode") == "production"

BUCKET_NAME = "cuny-sps-citibike-archive"
CHART_KEY = "exports/live/performance.png"

spark.conf.set("spark.sql.session.timeZone", "UTC")

# The defaults are the shared workspace's tables
dbutils.widgets.text("schema", "citibike_project.citibike")
dbutils.widgets.text("status_table", "citibike_project.citibike.silver_station_status")
dbutils.widgets.text("info_table", "citibike_project.citibike.silver_station_info")
dbutils.widgets.text("lookback_hours", "24")
SCHEMA = dbutils.widgets.get("schema")
STATUS_TABLE = dbutils.widgets.get("status_table")
INFO_TABLE = dbutils.widgets.get("info_table")
LOOKBACK_HOURS = int(dbutils.widgets.get("lookback_hours"))

PREDICTIONS_TABLE = f"{SCHEMA}.station_predictions"
LOCAL_TZ = "America/New_York"
HORIZON_MIN = 30
MAX_OBSERVATION_GAP_MIN = 10

### The newest predictions

In [0]:
all_predictions = spark.table(PREDICTIONS_TABLE)

newest_prediction_time = all_predictions.agg(F.max("prediction_time")).first()[0]
newest_predictions = (all_predictions
                      .filter(F.col("prediction_time") == F.lit(newest_prediction_time))
                      .toPandas())
new_york_prediction_time = pd.Timestamp(newest_prediction_time, tz="UTC").tz_convert(LOCAL_TZ)

prediction_age_minutes = (datetime.now(timezone.utc).replace(tzinfo=None) - newest_prediction_time).total_seconds() / 60
print(f"Newest predictions: {new_york_prediction_time:%Y-%m-%d %H:%M} New York time, {prediction_age_minutes:.0f} minutes ago")

station_locations = newest_predictions[["lat", "lon"]].dropna()
plot_prediction_maps(newest_predictions, new_york_prediction_time, station_locations)
plt.show()

### Recent performance

The outcomes are computed from silver with `build_targets`, the same rule as the training targets.

In [0]:
window_end = datetime.now(timezone.utc).replace(tzinfo=None)
window_start = window_end - timedelta(hours=LOOKBACK_HOURS)

# Outcomes: the polls of the window, plus one hour before it, so the first polls have their full history
recent_status = spark.table(STATUS_TABLE).filter(F.col("fetched_at") >= F.lit(window_start - timedelta(hours=1)))
station_info = spark.table(INFO_TABLE)
realized_outcomes = build_targets(
    build_predictors(recent_status, station_info, local_timezone=LOCAL_TZ, max_gap_minutes=MAX_OBSERVATION_GAP_MIN),
    horizon_minutes=HORIZON_MIN, max_gap_minutes=MAX_OBSERVATION_GAP_MIN,
)

# Each prediction row is for one station and one poll, so it joins to the outcome of that poll
predictions_with_outcomes = (
    all_predictions
    .filter(F.col("prediction_time") >= F.lit(window_start))
    .join(realized_outcomes.select("station_id", "fetched_at", "bike_target", "dock_target"),
          ["station_id", "fetched_at"], "inner")
)

prediction_hour = F.date_trunc("hour", F.from_utc_timestamp("prediction_time", LOCAL_TZ)).alias("prediction_hour")

bike_evaluations = (
    predictions_with_outcomes
    .filter((F.col("bike_state") == "eligible") & F.col("bike_target").isNotNull())
    .select(F.lit("bike").alias("model_type"), prediction_hour,
            F.col("p_bike_stockout").alias("predicted_probability"),
            F.col("bike_target").alias("observed"))
)
dock_evaluations = (
    predictions_with_outcomes
    .filter((F.col("dock_state") == "eligible") & F.col("dock_target").isNotNull())
    .select(F.lit("dock").alias("model_type"), prediction_hour,
            F.col("p_dock_stockout").alias("predicted_probability"),
            F.col("dock_target").alias("observed"))
)

evaluation_data = bike_evaluations.unionByName(dock_evaluations).toPandas()

# Clip only for numerical stability when taking logarithms
clipped_probability = evaluation_data.predicted_probability.clip(1e-12, 1 - 1e-12)
evaluation_data["log_loss"] = -(evaluation_data.observed * np.log(clipped_probability)
                                + (1 - evaluation_data.observed) * np.log(1 - clipped_probability))
print(f"{len(evaluation_data):,} evaluated predictions between "
      f"{evaluation_data.prediction_hour.min()} and {evaluation_data.prediction_hour.max()} (New York time)")

In [0]:
# Summary for the window. The baseline predicts the window's own observed rate for every station;
# it is a demanding benchmark, because it knows the average outcome in advance.

summary_rows = []
for model_type, model_type_data in evaluation_data.groupby("model_type"):
    event_rate = model_type_data.observed.mean()
    baseline_log_loss = -(event_rate * np.log(event_rate) + (1 - event_rate) * np.log(1 - event_rate))
    summary_rows.append({
        "model_type": model_type,
        "predictions": len(model_type_data),
        "event_rate": event_rate,
        "mean_prediction": model_type_data.predicted_probability.mean(),
        "log_loss": model_type_data.log_loss.mean(),
        "baseline_log_loss": baseline_log_loss,
        "skill": 1 - model_type_data.log_loss.mean() / baseline_log_loss,
        "roc_auc": roc_auc_score(model_type_data.observed, model_type_data.predicted_probability),
    })

display(pd.DataFrame(summary_rows).round(4))

In [0]:
# Hourly performance: log loss, and the observed against the predicted onset rate

hourly_performance = (
    evaluation_data
    .groupby(["model_type", "prediction_hour"])
    .agg(predictions=("observed", "size"), observed_rate=("observed", "mean"),
         predicted_rate=("predicted_probability", "mean"), log_loss=("log_loss", "mean"))
    .reset_index()
)

figure, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True, sharex=True)

for row, model_type in enumerate(["bike", "dock"]):
    model_type_hours = hourly_performance[hourly_performance.model_type == model_type]

    axis = axes[row, 0]
    axis.plot(model_type_hours.prediction_hour, model_type_hours.log_loss, "o-")
    axis.set(title=f"{model_type.title()}: log loss by hour (lower is better)", ylabel="Log loss")

    axis = axes[row, 1]
    axis.plot(model_type_hours.prediction_hour, 100 * model_type_hours.observed_rate, "o-", label="Observed")
    axis.plot(model_type_hours.prediction_hour, 100 * model_type_hours.predicted_rate, "--", label="Predicted")
    axis.set(title=f"{model_type.title()}: onset rate by hour", ylabel="Percent of eligible stations")
    axis.legend()

for axis in axes[1]:
    axis.set_xlabel("Hour (New York time)")
    axis.tick_params(axis="x", labelrotation=45)

add_figure_heading(figure, f"Prediction performance, last {LOOKBACK_HOURS} hours", panels_top=0.93)
plt.show()

In [0]:
# Calibration over the window: observed onset rate against mean predicted probability, in 20 bins.
# Points on the diagonal mean the probabilities can be taken at face value.

figure, axes = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)

for axis, model_type in zip(axes, ["bike", "dock"]):
    model_type_data = evaluation_data[evaluation_data.model_type == model_type]
    calibration = (
        model_type_data
        .assign(probability_bin=np.minimum((model_type_data.predicted_probability * 20).astype(int), 19))
        .groupby("probability_bin")
        .agg(predictions=("observed", "size"), predicted=("predicted_probability", "mean"),
             observed=("observed", "mean"))
        .reset_index()
    )
    axis.plot([0, 1], [0, 1], "--", color="gray")
    axis.scatter(calibration.predicted, calibration.observed, s=12 + 4 * np.sqrt(calibration.predictions))
    axis.set(title=f"{model_type.title()}: calibration", xlabel="Mean predicted probability",
             ylabel="Observed onset rate", xlim=(0, 1), ylim=(0, 1))

add_figure_heading(figure, f"Calibration, last {LOOKBACK_HOURS} hours (marker size: number of predictions)",
                   title_size=16, panels_top=0.9)
plt.show()

In [0]:
# The log loss by hour, as a small image for the public page
figure, axis = plt.subplots(figsize=(14, 3.2), constrained_layout=True)
for model_type, label in [("bike", "Stockout"), ("dock", "Dockout")]:
    model_type_hours = hourly_performance[hourly_performance.model_type == model_type]
    axis.plot(model_type_hours.prediction_hour, model_type_hours.log_loss, "o-", label=label)
axis.set(title=f"Log loss of the predictions by hour, last {LOOKBACK_HOURS} hours (lower is better)",
         ylabel="Log loss")
axis.legend()

if IS_PRODUCTION:
    chart_buffer = io.BytesIO()
    figure.savefig(chart_buffer, format="png", dpi=110)
    s3 = boto3.client(
        "s3",
        aws_access_key_id=dbutils.secrets.get("citibike-export", "aws-access-key-id"),
        aws_secret_access_key=dbutils.secrets.get("citibike-export", "aws-secret-access-key"),
        region_name="us-west-2",
    )
    s3.put_object(Bucket=BUCKET_NAME, Key=CHART_KEY, Body=chart_buffer.getvalue(),
                  ContentType="image/png", CacheControl="max-age=60")
    print(f"Published https://{BUCKET_NAME}.s3.us-west-2.amazonaws.com/{CHART_KEY}")
plt.show()